# exp01_pokemon1070front_condUNet-CFG — local generation (Mac / MPS)\n\nEarly (no attn) name-conditioned DDPM. Generate by name + SLERP fusion.


## Model + diffusion


In [ ]:
import os, json, math, torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision.utils import make_grid
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

OUT_DIR = "."                                   # the notebook is in the experiment folder
CKPT    = "checkpoints/ckpt_ep300.pt"
IMG_SIZE, TIMESTEPS, BASE = 96, 1000, 64
ATTN = False                                   # whether this experiment includes self-attn
print("device:", device, "| ATTN:", ATTN)

def sinusoidal_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / half)
    args = t[:, None].float() * freqs[None, :]
    return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

class ResBlock(nn.Module):
    def __init__(self, i, o, c, g=8):
        super().__init__()
        self.block1 = nn.Sequential(nn.GroupNorm(g, i), nn.SiLU(), nn.Conv2d(i, o, 3, padding=1))
        self.cond_mlp = nn.Sequential(nn.SiLU(), nn.Linear(c, o))
        self.block2 = nn.Sequential(nn.GroupNorm(g, o), nn.SiLU(), nn.Conv2d(o, o, 3, padding=1))
        self.skip = nn.Conv2d(i, o, 1) if i != o else nn.Identity()
    def forward(self, x, c):
        h = self.block1(x); h = h + self.cond_mlp(c)[:, :, None, None]; h = self.block2(h)
        return h + self.skip(x)

class AttnBlock(nn.Module):
    def __init__(self, ch, heads=4):
        super().__init__(); self.norm = nn.GroupNorm(8, ch); self.mha = nn.MultiheadAttention(ch, heads, batch_first=True)
    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x).reshape(B, C, H*W).transpose(1, 2); h, _ = self.mha(h, h, h)
        return x + h.transpose(1, 2).reshape(B, C, H, W)

class UNet(nn.Module):
    """Early flat structure: a single ResBlock per level, base=64, optional self-attn (added at down3/mid/up3)."""
    def __init__(self, base=64, c_dim=256, num_classes=None, attn=False):
        super().__init__()
        self.c_dim = c_dim; self.num_classes = num_classes; self.attn = attn
        self.time_mlp = nn.Sequential(nn.Linear(c_dim, c_dim), nn.SiLU(), nn.Linear(c_dim, c_dim))
        self.label_emb = nn.Embedding(num_classes + 1, c_dim)
        self.in_conv = nn.Conv2d(3, base, 3, padding=1)
        self.down1 = ResBlock(base, base, c_dim); self.down2 = ResBlock(base, base*2, c_dim); self.down3 = ResBlock(base*2, base*4, c_dim)
        if attn: self.down3_attn = AttnBlock(base*4)
        self.downsample = nn.AvgPool2d(2)
        if attn:
            self.mid1 = ResBlock(base*4, base*4, c_dim); self.mid_attn = AttnBlock(base*4); self.mid2 = ResBlock(base*4, base*4, c_dim)
        else:
            self.mid = ResBlock(base*4, base*4, c_dim)
        self.upsample = nn.Upsample(scale_factor=2, mode="nearest")
        self.up3 = ResBlock(base*4 + base*4, base*2, c_dim)
        if attn: self.up3_attn = AttnBlock(base*2)
        self.up2 = ResBlock(base*2 + base*2, base, c_dim); self.up1 = ResBlock(base + base, base, c_dim)
        self.out = nn.Sequential(nn.GroupNorm(8, base), nn.SiLU(), nn.Conv2d(base, 3, 3, padding=1))
    def forward(self, x, t, y=None, y_emb=None):
        c = self.time_mlp(sinusoidal_embedding(t, self.c_dim))
        if y_emb is not None:
            c = c + y_emb
        else:
            if y is None: y = torch.full((t.size(0),), self.num_classes, device=t.device, dtype=torch.long)
            c = c + self.label_emb(y)
        x = self.in_conv(x)
        s1 = self.down1(x, c); x = self.downsample(s1)
        s2 = self.down2(x, c); x = self.downsample(s2)
        s3 = self.down3(x, c)
        if self.attn: s3 = self.down3_attn(s3)
        x = self.downsample(s3)
        if self.attn:
            x = self.mid1(x, c); x = self.mid_attn(x); x = self.mid2(x, c)
        else:
            x = self.mid(x, c)
        x = self.upsample(x); x = self.up3(torch.cat([x, s3], 1), c)
        if self.attn: x = self.up3_attn(x)
        x = self.upsample(x); x = self.up2(torch.cat([x, s2], 1), c)
        x = self.upsample(x); x = self.up1(torch.cat([x, s1], 1), c)
        return self.out(x)

In [ ]:
class Diffusion:
    def __init__(self, timesteps=1000, beta_start=1e-4, beta_end=0.02, device="cpu"):
        self.T = timesteps; self.device = device
        beta = torch.linspace(beta_start, beta_end, timesteps, device=device)
        self.beta = beta; self.alpha = 1.0 - beta; self.alpha_bar = torch.cumprod(self.alpha, 0)
    @torch.no_grad()
    def sample(self, model, n, y=None, y_emb=None, guidance=3.0, img_size=96):
        model.eval()
        x = torch.randn(n, 3, img_size, img_size, device=self.device)
        use_cfg = (y is not None) or (y_emb is not None)
        null = torch.full((n,), model.num_classes, device=self.device, dtype=torch.long)
        for i in reversed(range(self.T)):
            t = torch.full((n,), i, device=self.device, dtype=torch.long)
            if use_cfg:
                eps_c = model(x, t, y=y, y_emb=y_emb); eps_u = model(x, t, y=null)
                pred = eps_u + guidance * (eps_c - eps_u)
            else:
                pred = model(x, t)
            bt, at, abt = self.beta[i], self.alpha[i], self.alpha_bar[i]
            mean = (1/at.sqrt()) * (x - (bt/(1-abt).sqrt()) * pred)
            x = mean + (bt.sqrt()*torch.randn_like(x) if i > 0 else 0.0)
        model.train(); return x

## Load weights + utilities


In [ ]:
classes = json.load(open(os.path.join(OUT_DIR, "classes.json"), encoding="utf-8"))
model = UNet(base=BASE, num_classes=len(classes), attn=ATTN).to(device)
model.load_state_dict(torch.load(CKPT, map_location=device)); model.eval()
diff = Diffusion(timesteps=TIMESTEPS, device=device)
print("loaded:", CKPT, "| num classes:", len(classes))

def find_idx(q):
    q = q.lower(); hits = [(i, n) for i, n in enumerate(classes) if q in n.lower()]
    if not hits: print("Not found:", q)
    return hits
def show(imgs, title=""):
    grid = make_grid(((imgs.clamp(-1,1)+1)/2).cpu(), nrow=min(len(imgs),4)).permute(1,2,0).numpy()
    plt.figure(figsize=(8,8)); plt.imshow(grid); plt.axis("off"); plt.title(title); plt.show()
def emb_of(name): return model.label_emb(torch.tensor([find_idx(name)[0][0]], device=device))
def slerp(a, b, t):
    a, b = a.squeeze(0), b.squeeze(0)
    om = torch.acos(((a/a.norm())*(b/b.norm())).sum().clamp(-1,1)); so = torch.sin(om)
    out = ((1-t)*a+t*b) if so < 1e-6 else (torch.sin((1-t)*om)/so)*a + (torch.sin(t*om)/so)*b
    return out.unsqueeze(0)

## 1. Generate by name


In [ ]:
NAME = "pikachu"     # ← change to what you want (fuzzy match)
N, GUIDANCE = 8, 4.0
idx = find_idx(NAME)[0][0]
y = torch.full((N,), idx, device=device, dtype=torch.long)
show(diff.sample(model, N, y=y, guidance=GUIDANCE, img_size=IMG_SIZE), f"{classes[idx]} x{N}")

## 2. Fusion (name-embedding SLERP interpolation)


In [ ]:
NAME_A, NAME_B = "charmander", "squirtle"   # ← the two
ALPHA, N, GUIDANCE = 0.5, 8, 4.0
fused = slerp(emb_of(NAME_A), emb_of(NAME_B), ALPHA).repeat(N, 1)   # SLERP norm-preserving
show(diff.sample(model, N, y_emb=fused, guidance=GUIDANCE, img_size=IMG_SIZE), f"{NAME_A} x {NAME_B} (a={ALPHA})")

## 3. Unconditional generation


In [ ]:
show(diff.sample(model, 8, img_size=IMG_SIZE), "unconditional generation")